In [1]:
# ============================================================
# PHASE 2 (Redesigned & Perfected): Feature Space Definition
# ============================================================
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, StandardScaler

# 1. Define Paths using os for cross-platform compatibility
notebook_dir = os.getcwd()
CSV_PATH = os.path.join(notebook_dir, "..", "data", "raw", "creditcard.csv")

# Define processed directory and ensure it exists
PROCESSED_DIR = os.path.join(notebook_dir, "..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 2. Load original data
print(f"Loading original ULB dataset from {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)

# 3. Create stable identifier and isolate label
df['row_id'] = np.arange(len(df))
y = df['Class'].copy()
X_raw = df.drop(columns=['Class'])

# Define features
v_cols = [c for c in X_raw.columns if c.startswith('V')]
scale_cols = ['Time', 'Amount']

# ==========================================
# GROUND TRUTH REGISTRY (Leakage Prevention)
# ==========================================
print("\nSaving Ground Truth Registry...")
truth_registry = pd.DataFrame({'row_id': df['row_id'], 'Class': y})
truth_registry.to_csv(os.path.join(PROCESSED_DIR, "ground_truth_registry.csv"), index=False)

# ==========================================
# FEATURE SPACE A: Baseline (RobustScaler)
# ==========================================
print("Building FS-A (Baseline: RobustScaler)...")
fs_a = X_raw.copy()

scaler_a = RobustScaler()
fs_a[scale_cols] = scaler_a.fit_transform(fs_a[scale_cols])

# Save strictly pure features + row_id
fs_a.to_csv(os.path.join(PROCESSED_DIR, "ULB_FS_A_Baseline.csv"), index=False)

# ==========================================
# FEATURE SPACE B: Amount Sensitivity (log1p + StandardScaler)
# ==========================================
print("Building FS-B (Sensitivity: log1p + StandardScaler)...")
fs_b = X_raw.copy()

# Apply log1p to Amount to handle extreme skew, then scale both Time & Amount using Standard
fs_b['Amount'] = np.log1p(fs_b['Amount'])
scaler_b = StandardScaler()
fs_b[scale_cols] = scaler_b.fit_transform(fs_b[scale_cols])

# Save strictly pure features + row_id
fs_b.to_csv(os.path.join(PROCESSED_DIR, "ULB_FS_B_LogAmount.csv"), index=False)

# ==========================================
# CREATE FEATURE SPACE MANIFEST
# ==========================================
manifest = """
# Phase 2 — Feature Space Manifest

## Dataset Integrity
- **Rows:** 284,807
- **Missing Values:** 0
- **Target (Class):** Isolated to `ground_truth_registry.csv` to prevent unsupervised leakage.

## Feature Taxonomy
- **Group A (V1-V28):** PCA-derived. Left strictly untouched.
- **Group B (Time):** Elapsed seconds. Scaled.
- **Group C (Amount):** Transaction value. Highly skewed (skew=16.98).

## Controlled Variants
| Feature Space | Features | Transformations | Scaling | Purpose |
|---|---|---|---|---|
| **FS-A** | V1-V28, Time, Amount | Baseline | RobustScaler (Time, Amount) | Primary baseline (median/IQR) |
| **FS-B** | V1-V28, Time, Amount | log1p(Amount) | StandardScaler (Time, Amount) | Sensitivity (variance stabilization) |

## Output Artefacts
1. `ground_truth_registry.csv` (row_id, Class)
2. `ULB_FS_A_Baseline.csv` (row_id, features)
3. `ULB_FS_B_LogAmount.csv` (row_id, features)
"""
with open(os.path.join(PROCESSED_DIR, "feature_space_manifest.md"), "w") as f:
    f.write(manifest)

print(f"\nManifest and datasets saved successfully to: {PROCESSED_DIR}")
print("Phase 2 Redesign Complete. Ready for Phase 3 restructuring.")

Loading original ULB dataset from c:\Users\53542\Desktop\mplad-anomaly-detection\notebooks\..\data\raw\creditcard.csv...

Saving Ground Truth Registry...
Building FS-A (Baseline: RobustScaler)...
Building FS-B (Sensitivity: log1p + StandardScaler)...

Manifest and datasets saved successfully to: c:\Users\53542\Desktop\mplad-anomaly-detection\notebooks\..\data\processed
Phase 2 Redesign Complete. Ready for Phase 3 restructuring.
